# P5 household power forecasting: Kaggle runner

Thin wrapper: all logic lives in `src/tsforecast`.

Settings: **Internet ON** (a GPU T4 speeds up the LSTM/Transformer but CPU works), then
*Save Version -> Save & Run All*. The run can be resumed: rerunning skips (model, seed) rows that already finished.

Download `p5_results.zip` from the Output tab at the end. It holds the metrics, error analysis, anomaly results,
figures and the test forecasts. Model weights and the raw data are left out.

In [ ]:
!git clone --depth 1 https://github.com/JDevender007/Artificial_Intelligence.git
%cd Artificial_Intelligence/05-load-forecasting
!pip install -q -r requirements-gpu.txt && pip install -q -e . --no-deps

In [ ]:
!mkdir -p /kaggle/working/outputs/household_power
!python -c "import torch, numpy, pandas, scipy, statsmodels; print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); print('numpy', numpy.__version__); print('pandas', pandas.__version__); print('scipy', scipy.__version__); print('statsmodels', statsmodels.__version__)" | tee /kaggle/working/outputs/household_power/env.txt
!(nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo no-gpu) | tee -a /kaggle/working/outputs/household_power/env.txt
!lscpu | grep 'Model name' | tee -a /kaggle/working/outputs/household_power/env.txt

## Run `configs/household_power.yaml`
Downloads the UCI file, cleans it to hourly, runs the walk-forward backtest for every model x seed, then the anomaly evaluation and error analysis.

In [ ]:
!python scripts/run_experiment.py --config configs/household_power.yaml --set output_dir=/kaggle/working/outputs/household_power

In [ ]:
from pathlib import Path

for f in ['data_summary.md', 'results.md', 'error_analysis.md', 'anomaly.md']:
    p = Path('/kaggle/working/outputs/household_power') / f
    print(f'===== {f}')
    print(p.read_text() if p.exists() else '(missing)')

## Package results
The zip holds text results, figures and forecasts only (no weights).

In [ ]:
%cd /kaggle/working
!zip -qr p5_results.zip outputs -x '*/models/*'
!rm -rf outputs Artificial_Intelligence && ls -la /kaggle/working